In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D9 — Estatística do Movimento Fisiológico da População
#      de Portugal — Ano de 1925
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

import json
import math
import re
import hashlib
import unicodedata

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd



In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"
INPUT_REPRESENTATION = "Original scanned PDF"

EXPECTED_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Topic",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

DESCRIPTION_DIAGNOSTIC_FIELD = "Description"

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location",
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Reporting Period",
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

REFERENCE_PATH = Path("D9_reference_values.csv")
EXTRACTION_PATH = Path("D9_branch_A_parsed_extraction.json")
TECHNICAL_DIAGNOSTICS_PATH = Path(
    "D9_branch_A_technical_diagnostics.json"
)
EXPERIMENT_METADATA_PATH = Path("D9_branch_A_experiment_metadata.json")

EXPECTED_SOURCE_SHA256 = (
    "a8fd4297949fa5b75b6e801ecd95aa9f7c15f9b08c789bc530f2266f41106cc1"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected reference records:", EXPECTED_RECORD_COUNT)
print("Fields:", FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:
# ============================================================
# 2. Upload canonical validation inputs
# ============================================================

try:
    from google.colab import files

    required_files = [
        REFERENCE_PATH.name,
        EXTRACTION_PATH.name,
        TECHNICAL_DIAGNOSTICS_PATH.name,
        EXPERIMENT_METADATA_PATH.name,
    ]

    missing_files = [
        filename
        for filename in required_files
        if not Path(filename).exists()
    ]

    if missing_files:
        print("Upload the following canonical D9 files:")
        for filename in missing_files:
            print(" -", filename)

        uploaded = files.upload()

        print("\nUploaded:")
        for filename in uploaded:
            print(" -", filename)
    else:
        print("All required inputs are already present.")

except ImportError:
    print("Not running in Google Colab.")
    print("Place the four required inputs in the working directory.")

for path in [
    REFERENCE_PATH,
    EXTRACTION_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")


In [ ]:
# ============================================================
# 3. File hashing
# ============================================================

def sha256_file(path):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(TECHNICAL_DIAGNOSTICS_PATH)
EXPERIMENT_METADATA_SHA256 = sha256_file(EXPERIMENT_METADATA_PATH)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Technical-diagnostics SHA-256:", TECHNICAL_DIAGNOSTICS_SHA256)
print("Experiment-metadata SHA-256:", EXPERIMENT_METADATA_SHA256)


In [ ]:
# ============================================================
# 4. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)


def restore_reference_value(value):
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        try:
            number = float(text)
            return int(number) if number.is_integer() else number
        except ValueError:
            pass

    return value


reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

print("Reference records:", len(reference_df))
print("Reference schema exact:", reference_schema_exact)

display(reference_df)

In [ ]:
# ============================================================
# 5. Load canonical preserved Branch A extraction
# ============================================================

with EXTRACTION_PATH.open("r", encoding="utf-8") as file:
    parsed_extraction = json.load(file)

valid_json = True
top_level_object_valid = isinstance(parsed_extraction, dict)

document_id_correct = (
    top_level_object_valid
    and parsed_extraction.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and parsed_extraction.get("branch") == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(parsed_extraction.get("records"), list)
)

if not records_is_list:
    raise TypeError(
        "Canonical D9 Branch A parsed extraction has no valid records list."
    )

extracted_records = parsed_extraction["records"]
extracted_df = pd.DataFrame(extracted_records)

print("Top-level object valid:", top_level_object_valid)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Extracted records:", len(extracted_records))

display(extracted_df)


In [ ]:
# ============================================================
# 6. Load Branch A technical diagnostics and experiment metadata
# ============================================================

with TECHNICAL_DIAGNOSTICS_PATH.open("r", encoding="utf-8") as file:
    technical_diagnostics = json.load(file)

with EXPERIMENT_METADATA_PATH.open("r", encoding="utf-8") as file:
    experiment_metadata = json.load(file)

branch_a_structurally_evaluable = bool(
    technical_diagnostics.get("structurally_evaluable")
)

metadata_parsed_hash = experiment_metadata.get(
    "parsed_extraction_sha256"
)

parsed_extraction_hash_matches_metadata = (
    metadata_parsed_hash == EXTRACTION_SHA256
)

metadata_source_hash = experiment_metadata.get(
    "source_sha256"
)

source_hash_matches_stage_1 = (
    metadata_source_hash == EXPECTED_SOURCE_SHA256
)

print("Branch A structure valid:", branch_a_structurally_evaluable)
print(
    "Parsed extraction hash matches metadata:",
    parsed_extraction_hash_matches_metadata,
)
print(
    "Source hash matches Stage 1:",
    source_hash_matches_stage_1,
)

if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "The extraction file does not match Branch A experiment metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "The Branch A source hash does not match the fixed Stage 1 source."
    )


In [ ]:
# ============================================================
# 7. Schema validation
# ============================================================

schema_issue_rows = []
field_order_diagnostic_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Record is not a JSON object",
        })
        continue

    observed_fields = list(record.keys())
    observed_set = set(observed_fields)
    expected_set = set(FIELDS)

    missing_fields = [
        field for field in FIELDS
        if field not in observed_set
    ]

    extra_fields = [
        field for field in observed_fields
        if field not in expected_set
    ]

    if missing_fields or extra_fields:
        schema_issue_rows.append({
            "Record Index": record_index,
            "Issue": "Field set mismatch",
            "Missing Fields": ", ".join(missing_fields),
            "Extra Fields": ", ".join(extra_fields),
        })

    if observed_fields != FIELDS:
        field_order_diagnostic_rows.append({
            "Record Index": record_index,
            "Observed Order": observed_fields,
            "Expected Order": FIELDS,
        })


schema_issues_df = pd.DataFrame(schema_issue_rows)
field_order_diagnostics_df = pd.DataFrame(
    field_order_diagnostic_rows
)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issues:", len(schema_issues_df))
print(
    "Field-order diagnostic count:",
    len(field_order_diagnostics_df),
)

In [ ]:
# ============================================================
# 8. Type and mandatory-content diagnostics
# ============================================================

type_issue_rows = []
missing_mandatory_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        continue

    for field in MANDATORY_STRING_FIELDS:
        value = record.get(field)

        if value is None or value == "":
            missing_mandatory_rows.append({
                "Record Index": record_index,
                "Field": field,
            })

        elif not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string",
            })

    for field in NULLABLE_STRING_FIELDS:
        value = record.get(field)

        if value is not None and not isinstance(value, str):
            type_issue_rows.append({
                "Record Index": record_index,
                "Field": field,
                "Observed Type": type(value).__name__,
                "Expected Type": "string or null",
            })

    value = record.get("Value")

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (str, int, float))
        )
    ):
        type_issue_rows.append({
            "Record Index": record_index,
            "Field": "Value",
            "Observed Type": type(value).__name__,
            "Expected Type": "string, number or null",
        })


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_fields_df = pd.DataFrame(
    missing_mandatory_rows
)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_fields_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)

In [ ]:
# ============================================================
# 9. Reference integrity and extraction content diagnostics
# ============================================================

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

extracted_record_count = len(extracted_records)

extraction_record_count_valid = (
    extracted_record_count == EXPECTED_RECORD_COUNT
)

extraction_category_counts = dict(
    Counter(
        record.get("Category")
        for record in extracted_records
        if isinstance(record, dict)
    )
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

print("Reference record count valid:", reference_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction record count valid:", extraction_record_count_valid)
print("Extraction category counts valid:", extraction_category_counts_valid)

if not reference_record_count_valid:
    raise AssertionError("Fixed D9 reference must contain 19 records.")

if not reference_category_counts_valid:
    raise AssertionError(
        "Fixed D9 reference category distribution is invalid."
    )

In [ ]:
# ============================================================
# 10. Comparison-only normalisation
# ============================================================

def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalise_text(value):
    if is_missing(value):
        return None

    text = str(value)

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Cf"
    )

    text = unicodedata.normalize("NFKC", text)

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00a0", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text.casefold()


def identity_text(value):
    text = normalise_text(value)

    if text is None:
        return ""

    text = re.sub(r"[^a-z0-9à-ÿ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def lexical_similarity(first, second):
    from difflib import SequenceMatcher

    first_text = normalise_text(first) or ""
    second_text = normalise_text(second) or ""

    if not first_text and not second_text:
        return 1.0

    if not first_text or not second_text:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text,
    ).ratio()


def exact_normalised_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


In [ ]:
# ============================================================
# 11. Conservative controlled comparison rules
# ============================================================

UNIT_EQUIVALENCE_MAP = {
    "year": "year",
    "square kilometres": "square kilometres",
    "square kilometers": "square kilometres",
    "km²": "square kilometres",
    "km2": "square kilometres",
    "people": "people",
    "persons": "people",
    "inhabitants per square kilometre":
        "inhabitants per square kilometre",
    "inhabitants per square kilometer":
        "inhabitants per square kilometre",
    "per thousand inhabitants":
        "per thousand inhabitants",
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(text, text)


def canonical_reporting_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    # Safe notation equivalence only.
    text = re.sub(r"\s*-\s*", "-", text)

    return text


def canonical_source_location(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = re.sub(r"^physical\s+", "", text)
    text = text.replace("índice", "index")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def unit_correct(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def period_correct(reference_value, extracted_value):
    return (
        canonical_reporting_period(reference_value)
        == canonical_reporting_period(extracted_value)
    )


def source_location_correct(reference_value, extracted_value):
    return (
        canonical_source_location(reference_value)
        == canonical_source_location(extracted_value)
    )


print(
    "Conservative D9 comparison rules loaded. "
    "No Branch-A-derived semantic equivalence rules are active."
)


In [ ]:
# ============================================================
# 12. Type-aware Value comparison
# ============================================================

def numeric_value(value):
    if is_missing(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return None


def value_correct(reference_value, extracted_value):

    if is_missing(reference_value) and is_missing(extracted_value):
        return True

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    if isinstance(reference_value, str) and isinstance(extracted_value, str):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    return False


In [ ]:
# ============================================================
# 13. Fixed identity keys preparation
# ============================================================

reference_comparison_df = reference_df.copy()
extracted_comparison_df = pd.DataFrame(extracted_records).copy()

reference_comparison_df["_reference_index"] = np.arange(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = np.arange(
    len(extracted_comparison_df)
)

for frame in [
    reference_comparison_df,
    extracted_comparison_df,
]:
    frame["_identity_category"] = frame["Category"].map(
        identity_text
    )

    frame["_identity_topic"] = frame["Topic"].map(
        identity_text
    )

    frame["_identity_key"] = list(zip(
        frame["_identity_category"],
        frame["_identity_topic"],
    ))


reference_duplicate_identity_count = int(
    reference_comparison_df["_identity_key"].duplicated().sum()
)

extraction_duplicate_identity_count = int(
    extracted_comparison_df["_identity_key"].duplicated().sum()
)

print(
    "Reference duplicate Category+Topic identities:",
    reference_duplicate_identity_count,
)

print(
    "Extraction duplicate Category+Topic identities:",
    extraction_duplicate_identity_count,
)

if reference_duplicate_identity_count:
    raise AssertionError(
        "The fixed D9 reference does not have unique Category+Topic identities."
    )


In [ ]:
# ============================================================
# 14. One-to-one outcome-independent alignment
# ============================================================

extraction_key_to_indices = {}

for _, row in extracted_comparison_df.iterrows():
    extraction_key_to_indices.setdefault(
        row["_identity_key"],
        []
    ).append(int(row["_extraction_index"]))


matched_pairs = []
matched_extraction_indices = set()
missing_reference_indices = []

for _, reference_row in reference_comparison_df.iterrows():

    reference_index = int(
        reference_row["_reference_index"]
    )

    key = reference_row["_identity_key"]

    available_indices = [
        index
        for index in extraction_key_to_indices.get(key, [])
        if index not in matched_extraction_indices
    ]

    if len(available_indices) == 1:
        extraction_index = available_indices[0]

        matched_pairs.append({
            "reference_index": reference_index,
            "extraction_index": extraction_index,
        })

        matched_extraction_indices.add(
            extraction_index
        )

    else:
        missing_reference_indices.append(
            reference_index
        )


all_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].astype(int)
)

unsupported_extraction_indices = sorted(
    all_extraction_indices - matched_extraction_indices
)

missing_reference_indices = sorted(
    missing_reference_indices
)

print("Aligned records:", len(matched_pairs))
print("Missing reference records:", len(missing_reference_indices))
print(
    "Unsupported/unmatched extracted records:",
    len(unsupported_extraction_indices),
)


In [ ]:
# ============================================================
# 15. Missing and unsupported/unmatched record tables
# ============================================================

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            missing_reference_indices
        ),
        FIELDS + ["_reference_index"],
    ]
    .rename(columns={"_reference_index": "Reference Index"})
    .reset_index(drop=True)
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unsupported_extraction_indices
        ),
        FIELDS + ["_extraction_index"],
    ]
    .rename(columns={"_extraction_index": "Extraction Index"})
    .reset_index(drop=True)
)

print("Missing records:", len(missing_records_df))
print(
    "Unsupported/unmatched records:",
    len(unsupported_records_df),
)


In [ ]:
# ============================================================
# 16. Field-level comparison of aligned records
# ============================================================

comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    field_matches = {
        "Category": exact_normalised_text_match(
            reference_row["Category"],
            extracted_row["Category"],
        ),

        "Topic": exact_normalised_text_match(
            reference_row["Topic"],
            extracted_row["Topic"],
        ),

        "Description": exact_normalised_text_match(
            reference_row["Description"],
            extracted_row["Description"],
        ),

        "Value": value_correct(
            reference_row["Value"],
            extracted_row["Value"],
        ),

        "Unit": unit_correct(
            reference_row["Unit"],
            extracted_row["Unit"],
        ),

        "Reporting Period": period_correct(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"],
        ),

        "Source Location": source_location_correct(
            reference_row["Source Location"],
            extracted_row["Source Location"],
        ),
    }

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    all_primary_fields_match = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        field_matches[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    identity_label_difference = (
        all_primary_fields_match
        and not identity_fields_match
    )

    fully_correct = all_primary_fields_match

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Category": reference_row["Category"],
        "Topic": reference_row["Topic"],
        "Description Lexical Similarity": lexical_similarity(
            reference_row["Description"],
            extracted_row["Description"],
        ),
        "Fully Correct": bool(fully_correct),
        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),
        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields),
        "all_primary_fields_match":
            bool(all_primary_fields_match),

        "identity_fields_match":
            bool(identity_fields_match),

        "identity_label_difference":
            bool(identity_label_difference),
    }

    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(
            field_matches[field]
        )

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))
print(
    "Fully correct primary records:",
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0,
)

display(comparison_df)


In [ ]:
# ============================================================
# 17. Split fully correct and discrepant records
# ============================================================

if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()
else:
    fully_correct_records_df = comparison_df.loc[
        comparison_df["Fully Correct"]
    ].copy()

    discrepant_records_df = comparison_df.loc[
        ~comparison_df["Fully Correct"]
    ].copy()

print("Fully correct:", len(fully_correct_records_df))
print("Discrepant:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "Topic",
                "primary_mismatched_fields",
            ]
        ].reset_index(drop=True)
    )


In [ ]:
# ============================================================
# 18. Field-level accuracy
# ============================================================

field_rows = []

for field in PRIMARY_CORRECTNESS_FIELDS:

    evaluated_count = len(comparison_df)

    correct_count = (
        int(
            comparison_df[
                f"{field} Match"
            ].sum()
        )
        if evaluated_count > 0
        else 0
    )

    accuracy = (
        correct_count / evaluated_count
        if evaluated_count > 0
        else None
    )

    field_rows.append({
        "Field": field,
        "Aligned Records":
            evaluated_count,
        "Correct Records":
            correct_count,
        "Incorrect Records":
            evaluated_count - correct_count,
        "Accuracy":
            accuracy,
    })

field_validation_df = pd.DataFrame(
    field_rows
)

field_error_summary_df = field_validation_df.loc[
    field_validation_df["Incorrect Records"] > 0
].copy()

display(field_validation_df)


In [ ]:
# ============================================================
# 19. Record-level metrics
# ============================================================

reference_record_count = len(reference_df)
extracted_record_count = len(extracted_records)
aligned_record_count = len(comparison_df)

fully_correct_record_count = (
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0
)

discrepant_record_count = (
    aligned_record_count - fully_correct_record_count
)

missing_record_count = len(missing_records_df)
unsupported_record_count = len(unsupported_records_df)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if record_precision_exact + record_recall_exact > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

primary_correct_count = int(
    field_validation_df[
        "Correct Records"
    ].sum()
)

primary_total_count = int(
    field_validation_df[
        "Aligned Records"
    ].sum()
)

field_accuracy = (
    primary_correct_count / primary_total_count
    if primary_total_count > 0
    else None
)

description_diagnostic_accuracy = (
    float(
        comparison_df[
            "Description Match"
        ].mean()
    )
    if not comparison_df.empty
    else None
)

print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Field accuracy:",
    None
    if field_accuracy is None
    else round(field_accuracy, 4),
)
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)


In [ ]:
# ============================================================
# 20. Category-level metrics
# ============================================================

category_rows = []

for category in EXPECTED_CATEGORY_COUNTS:

    expected_records = int(
        (reference_df["Category"] == category).sum()
    )

    extracted_records_category = sum(
        1
        for record in extracted_records
        if record.get("Category") == category
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Category"] == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_category = len(category_comparison)

    fully_correct_category = (
        int(category_comparison["Fully Correct"].sum())
        if not category_comparison.empty
        else 0
    )

    discrepant_category = (
        aligned_records_category - fully_correct_category
    )

    category_completeness = (
        aligned_records_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_precision = (
        fully_correct_category / extracted_records_category
        if extracted_records_category > 0
        else 0.0
    )

    category_recall = (
        fully_correct_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall > 0
        else 0.0
    )

    category_rows.append({
        "Category": category,
        "Expected Records": expected_records,
        "Extracted Records": extracted_records_category,
        "Aligned Records": aligned_records_category,
        "Fully Correct Records": fully_correct_category,
        "Discrepant Records": discrepant_category,
        "Completeness": category_completeness,
        "Record Precision Exact": category_precision,
        "Record Recall Exact": category_recall,
        "Record F1 Exact": category_f1,
    })


category_metrics_df = pd.DataFrame(category_rows)
display(category_metrics_df)


In [ ]:
# ============================================================
# 21. Schema validity independent from completeness
# ============================================================

schema_validity = bool(
    branch_a_structurally_evaluable
)

schema_diagnostics = {
    "valid_json":
        bool(valid_json),
    "top_level_object_valid":
        bool(top_level_object_valid),
    "document_id_correct":
        bool(document_id_correct),
    "branch_correct":
        bool(branch_correct),
    "records_is_list":
        bool(records_is_list),
    "local_record_schema_valid":
        bool(record_schema_valid),
    "local_field_types_valid":
        bool(field_types_valid),
    "structurally_evaluable":
        bool(branch_a_structurally_evaluable),
}

content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "mandatory_fields_complete":
        bool(mandatory_fields_complete),
}

print("Schema validity:", schema_validity)
print(json.dumps(schema_diagnostics, indent=2))
print(
    reference_df.loc[
        reference_df["Category"] == "Index entry",
        ["Topic", "Value", "Reporting Period"]
    ]
)

print(
    reference_df.loc[
        reference_df["Category"] == "Document structure",
        ["Topic", "Value", "Reporting Period"]
    ]
)

print(
    reference_df.loc[
        reference_df["Topic"] == "Portugal area",
        ["Topic", "Value", "Reporting Period"]
    ]
)

In [ ]:
# ============================================================
# 22. D9 corrected-reference integrity confirmation
# ============================================================

reference_semantic_checks = {
    "index_entry_values_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Index entry",
            "Value"
        ].isna().all()
    ),

    "document_structure_values_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Document structure",
            "Value"
        ].isna().all()
    ),

    "document_structure_periods_null": bool(
        reference_df.loc[
            reference_df["Category"] == "Document structure",
            "Reporting Period"
        ].isna().all()
    ),

    "portugal_area_period_null": bool(
        pd.isna(
            reference_df.loc[
                reference_df["Topic"] == "Portugal area",
                "Reporting Period"
            ].iloc[0]
        )
    ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "Corrected D9 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D9 reference does not match the "
        "corrected frozen Stage 1 reference semantics."
    )

In [ ]:
# ============================================================
# 23. Summary table
# ============================================================

summary_table = pd.DataFrame([
    {"Metric": "Reference records", "Value": reference_record_count},
    {"Metric": "Extracted records", "Value": extracted_record_count},
    {"Metric": "Aligned records", "Value": aligned_record_count},
    {"Metric": "Fully correct records", "Value": fully_correct_record_count},
    {"Metric": "Discrepant records", "Value": discrepant_record_count},
    {"Metric": "Missing records", "Value": missing_record_count},
    {"Metric": "Unsupported/unmatched records", "Value": unsupported_record_count},
    {"Metric": "Completeness", "Value": completeness},
    {"Metric": "Exact record F1", "Value": record_f1_exact},
    {"Metric": "Field accuracy", "Value": field_accuracy},
    {"Metric": "Description diagnostic accuracy", "Value": description_diagnostic_accuracy},
    {"Metric": "Schema valid", "Value": schema_validity},
])

display(summary_table)


In [ ]:
# ============================================================
# 24. Create reproducible validation metrics
# ============================================================

field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records": int(row["Expected Records"]),
        "extracted_records": int(row["Extracted Records"]),
        "aligned_records": int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness": float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"]),
    }
    for _, row in category_metrics_df.iterrows()
}

VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),

    "completeness": round(float(completeness), 4),
    "missing_rate": round(float(missing_rate), 4),
    "record_precision_exact": round(float(record_precision_exact), 4),
    "record_recall_exact": round(float(record_recall_exact), 4),
    "record_f1_exact": round(float(record_f1_exact), 4),
    "unsupported_rate": round(float(unsupported_rate), 4),
    "discrepancy_rate_among_aligned":
        round(float(discrepancy_rate_among_aligned), 4),

    "field_accuracy": (
        None
        if field_accuracy is None
        else round(float(field_accuracy), 4)
    ),

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,


    "description_diagnostic_accuracy": (
        None
        if description_diagnostic_accuracy is None
        else round(float(description_diagnostic_accuracy), 4)
    ),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics":
        content_diagnostics,

    "matching_rules": {
        "identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,
        "one_to_one_assignment":
            "Unique deterministic Category + Topic identity",
        "value_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "reporting_period_used_for_alignment": False,
        "description_used_for_alignment": False,
        "source_location_used_for_alignment": False,
    },

    "comparison_rules": {
        "raw_extraction_modified": False,
        "manual_correction_applied": False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "numeric_comparison":
            "Exact represented numeric equality after deterministic parsing",
        "text_value_comparison":
            "Normalised exact textual equality; no fuzzy correctness",
        "description":
            (
                "Exact-string diagnostic plus lexical similarity diagnostic; "
                "excluded from primary exact-record correctness because the "
                "task permits a concise source-grounded description"
            ),
        "unit":
            "Controlled notation equivalence only",
        "reporting_period":
            "Normalised exact correctness with dash-notation normalisation only",
        "source_location":
            (
                "Normalised physical/source provenance; 'Physical PDF' prefix "
                "and Índice/Index terminology treated as equivalent"
            ),
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "d9_equivalence_rules_status":
            (
                "Frozen D9 document/schema-level comparison rules. "
                "No additional Branch-A-derived semantic equivalence rules "
                "were required. Reuse unchanged for Branches A, B and C."
            )
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid": bool(reference_semantics_valid),
        "checks": reference_semantic_checks,
        "reference_modified_by_validation": False,
    },

    "category_metrics":
        category_metrics_dictionary,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,
        "reference_sha256":
            REFERENCE_SHA256,
        "parsed_extraction_file":
            EXTRACTION_PATH.name,
        "parsed_extraction_sha256":
            EXTRACTION_SHA256,
        "technical_diagnostics_file":
            TECHNICAL_DIAGNOSTICS_PATH.name,
        "technical_diagnostics_sha256":
            TECHNICAL_DIAGNOSTICS_SHA256,
        "experiment_metadata_file":
            EXPERIMENT_METADATA_PATH.name,
        "experiment_metadata_sha256":
            EXPERIMENT_METADATA_SHA256,
        "branch_a_structurally_evaluable":
            bool(branch_a_structurally_evaluable),
        "parsed_extraction_hash_matches_metadata":
            bool(parsed_extraction_hash_matches_metadata),
        "source_hash_matches_stage_1":
            bool(source_hash_matches_stage_1),
    },


}

print(json.dumps(
    VALIDATION_METRICS,
    ensure_ascii=False,
    indent=2,
))


In [ ]:
# ============================================================
# 25. Validation metadata and conclusion
# ============================================================

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "validation_type":
        "Post-extraction reference-value agreement",
    "reference_dataset":
        REFERENCE_PATH.name,
    "extraction_dataset":
        EXTRACTION_PATH.name,
    "framework":
        (
            "Fixed corrected Stage 1 reference -> "
            "canonical Branch A extraction -> "
            "provenance/schema checks -> "
            "comparison-only normalisation -> "
            "outcome-independent Category+Topic alignment -> "
            "field comparison -> record classification -> "
            "common metrics -> corrected-reference integrity confirmation"
        ),
    "description_primary_correctness":
        False,
    "reference_modified":
        False,
}

VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "schema_valid":
        bool(schema_validity),
    "content_evaluable":
        True,
    "validation_completed":
        True,
    "equivalence_rules_frozen":
        True,
    "record_f1_exact":
        round(float(record_f1_exact), 4),

    "field_accuracy": (
        None
        if field_accuracy is None
        else round(float(field_accuracy), 4)
    ),

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,
    "next_step":
        (
            "Inspect the D9 discrepant records and the reference-audit table. "
            "Determine whether each discrepancy is an extraction error, a safe "
            "representation equivalence, or a source/schema inconsistency in "
            "the fixed Stage 1 reference. Do not add performance-driven rules."
        ),
}


In [ ]:
# ============================================================
# 26. Define output paths and export
# ============================================================

OUTPUT_PATHS = {
    "summary_json":
        Path("D9_branch_A_validation_summary.json"),
    "summary_csv":
        Path("D9_branch_A_validation_summary.csv"),
    "detailed_csv":
        Path("D9_branch_A_validation_detailed.csv"),
    "fully_correct_csv":
        Path("D9_branch_A_fully_correct_records.csv"),
    "discrepant_csv":
        Path("D9_branch_A_discrepant_records.csv"),
    "missing_csv":
        Path("D9_branch_A_missing_records.csv"),
    "unsupported_csv":
        Path("D9_branch_A_unsupported_records.csv"),
    "schema_issues_csv":
        Path("D9_branch_A_schema_issues.csv"),
    "field_order_csv":
        Path("D9_branch_A_field_order_diagnostics.csv"),
    "type_issues_csv":
        Path("D9_branch_A_type_issues.csv"),
    "missing_mandatory_csv":
        Path("D9_branch_A_missing_mandatory_fields.csv"),
    "field_validation_csv":
        Path("D9_branch_A_field_validation.csv"),
    "field_error_summary_csv":
        Path("D9_branch_A_field_error_summary.csv"),
    "category_metrics_csv":
        Path("D9_branch_A_category_metrics.csv"),
    "metadata_json":
        Path("D9_branch_A_validation_metadata.json"),
    "conclusion_json":
        Path("D9_branch_A_validation_conclusion.json"),
}

with OUTPUT_PATHS["summary_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_METRICS,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

summary_table.to_csv(
    OUTPUT_PATHS["summary_csv"],
    index=False,
    encoding="utf-8-sig",
)

comparison_df.to_csv(
    OUTPUT_PATHS["detailed_csv"],
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    OUTPUT_PATHS["fully_correct_csv"],
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    OUTPUT_PATHS["discrepant_csv"],
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    OUTPUT_PATHS["missing_csv"],
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    OUTPUT_PATHS["unsupported_csv"],
    index=False,
    encoding="utf-8-sig",
)

schema_issues_df.to_csv(
    OUTPUT_PATHS["schema_issues_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_order_diagnostics_df.to_csv(
    OUTPUT_PATHS["field_order_csv"],
    index=False,
    encoding="utf-8-sig",
)

type_issues_df.to_csv(
    OUTPUT_PATHS["type_issues_csv"],
    index=False,
    encoding="utf-8-sig",
)

missing_mandatory_fields_df.to_csv(
    OUTPUT_PATHS["missing_mandatory_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    OUTPUT_PATHS["field_validation_csv"],
    index=False,
    encoding="utf-8-sig",
)

field_error_summary_df.to_csv(
    OUTPUT_PATHS["field_error_summary_csv"],
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    OUTPUT_PATHS["category_metrics_csv"],
    index=False,
    encoding="utf-8-sig",
)

with OUTPUT_PATHS["metadata_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_METADATA,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

with OUTPUT_PATHS["conclusion_json"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        VALIDATION_CONCLUSION,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

print("D9 Validation A outputs exported.")


In [ ]:
# ============================================================
# 27. Final consistency checks
# ============================================================

assert reference_schema_exact
assert reference_record_count_valid
assert reference_category_counts_valid
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert all(path.exists() for path in OUTPUT_PATHS.values())
assert reference_semantics_valid

assert schema_validity == bool(
    branch_a_structurally_evaluable
)

assert (
    fully_correct_record_count
    + discrepant_record_count
    == aligned_record_count
)

assert (
    aligned_record_count + missing_record_count
    == reference_record_count
)

assert (
    aligned_record_count + unsupported_record_count
    == extracted_record_count
)

print("D9 Validation A revised notebook completed successfully.")
print("Schema valid:", schema_validity)
print("Aligned:", aligned_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Field accuracy:",
    None
    if field_accuracy is None
    else round(field_accuracy, 4),
)
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)
print(
    "Corrected reference semantics valid:",
    reference_semantics_valid,
)


In [ ]:
# ============================================================
# 28. List generated outputs
# ============================================================

print("Generated files:")

for label, path in OUTPUT_PATHS.items():
    print(
        f" - {label}: {path.name} "
        f"| exists: {path.exists()}"
    )
